In [1]:
#!pip install pyodbc sqlalchemy

In [2]:
import pandas as pd
import numpy as np

# Load the data directly from the zip to show efficient workflow
# Replace 'retail_store_sales.zip' with your actual filename
df = pd.read_csv(r'C:\Users\laksh\OneDrive\AI_Portfolio_Projects\Articulate_Projects\Rise_360_Project2\Sales_Automation_Project\Dataset\retail_store_sales.csv')

# Preview the 'Dirty' data
print("Raw Data Preview:")
df.head()

# Check for missing values and data types
print("--- Missing Values Report ---")
print(df.isnull().sum())

print("\n--- Data Types ---")
print(df.dtypes)

# 1. Convert Date to Datetime objects for time-series analysis
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'])

# 2. Categorical Cleaning: Fill missing Items
df['Item'] = df['Item'].fillna('General Merchandise')

# 3. Intelligent Imputation: Fill missing prices with the Average of that Category
df['Price Per Unit'] = df['Price Per Unit'].fillna(df.groupby('Category')['Price Per Unit'].transform('mean'))

# 4. Mathematical Consistency: Recalculate 'Total Spent'
df['Total Spent'] = df['Price Per Unit'] * df['Quantity'].fillna(1)

# 5. Boolean Logic: Assume missing discounts mean 'False'
df['Discount Applied'] = df['Discount Applied'].fillna(False)

print("Cleaning Complete. New Null Count:")
print(df.isnull().sum())

# Save the cleaned DataFrame to an Excel file
# index=False ensures we don't include those extra row numbers (0, 1, 2...)
df.to_excel(r'C:\Users\laksh\OneDrive\AI_Portfolio_Projects\Articulate_Projects\Rise_360_Project2\Sales_Automation_Project\Dataset\Cleaned_Sales_Data.xlsx', index=False, sheet_name='Sales Data')

print("Success! Your cleaned data has been exported to Excel.")

Raw Data Preview:
--- Missing Values Report ---
Transaction ID         0
Customer ID            0
Category               0
Item                1213
Price Per Unit       609
Quantity             604
Total Spent          604
Payment Method         0
Location               0
Transaction Date       0
Discount Applied    4199
dtype: int64

--- Data Types ---
Transaction ID       object
Customer ID          object
Category             object
Item                 object
Price Per Unit      float64
Quantity            float64
Total Spent         float64
Payment Method       object
Location             object
Transaction Date     object
Discount Applied     object
dtype: object
Cleaning Complete. New Null Count:
Transaction ID        0
Customer ID           0
Category              0
Item                  0
Price Per Unit        0
Quantity            604
Total Spent           0
Payment Method        0
Location              0
Transaction Date      0
Discount Applied      0
dtype: int64


C:\Users\laksh\AppData\Local\Temp\ipykernel_7464\1719028704.py:32: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Discount Applied'] = df['Discount Applied'].fillna(False)


Success! Your cleaned data has been exported to Excel.


In [3]:
import pandas as pd
import sqlalchemy
import urllib

# 1. DEFINING THE EXCEL FILE PATH
# The 'r' prefix handles the Windows backslashes safely
excel_file_path = r'C:\Users\laksh\OneDrive\AI_Portfolio_Projects\Articulate_Projects\Rise_360_Project2\Sales_Automation_Project\Dataset\Cleaned_Sales_Data.xlsx'

try:
    print("Reading cleaned Excel file...")
    # Changed from read_csv to read_excel
    df = pd.read_excel(excel_file_path) 
    print(f"Successfully loaded {len(df)} rows from Excel.")
except Exception as e:
    print(f"Error reading the Excel file: {e}")
    print("Make sure you install openpyxl if prompted (pip install openpyxl)")
    exit(1)

# 2. SETTING UP THE LOCAL SQL SERVER CONNECTION (CORRECTED)
params = urllib.parse.quote_plus(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=localhost;'              # <--- Changed from DESKTOP-705R5JE\laksh to just localhost
    'DATABASE=Demo;'                 
    'Trusted_Connection=yes;'
)
engine = sqlalchemy.create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# 3. PUSHING THE DATA TO THE SQL SERVER TABLE
try:
    print("Pushing data to SQL Server table 'sales_records'...")
    df.to_sql(
        name='sales_records', 
        con=engine, 
        if_exists='replace',  # Drops the table and recreates it. Change to 'append' if you want to keep existing data.
        index=False
    )
    print("--- Success! Excel data converted and pushed to SQL Server via shared memory ---")
except Exception as e:
    print(f"Database error: {e}")

Reading cleaned Excel file...
Successfully loaded 12575 rows from Excel.
Pushing data to SQL Server table 'sales_records'...
--- Success! Excel data converted and pushed to SQL Server via shared memory ---
